# Fraud Detection - XGBoost


In this notebook, we train an XGBoost classifier on the credit card dataset retrieved from Kaggle (https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud). 

The XGBoost classifier achieved a **precision of 53.6% and recall of 71.43%** on the holdout test set, and a **precision of 93.18% and recall of 83.67%** on the validation set. 

| | Validation | Test |
|---|---|---|
| Precision | 0.932 | 0.714 |
| Recall | 0.837 | 0.536 |

Notably, the model only produced 6 false-negatives in the unseen test set, reflecting the cost-conscious parameterization -- deliberately making the tradeoff of accepting higher false-positives to minimize the higher costs associated with false-negatives.


<br>

![Confusion Matrix](confusion_matrix.png)
<br>

The following techniques are used to build the XGBoost model:
* `TimeSeriesSplit` for selecting the Train/Val/Test splits, to prevent temporal data leakage (i.e. predicting the past with the future)
* Manual initialization of XGBoost's `scale_pos_weight` parameter value to compensate for class imbalances
* Hyperparameter tuning with Optuna TPE algorithm, setting eval metric to `aucpr` 
* Applying a cost-minimization approach to choosing the classifier threshold value, taking into account the cost-asymmetries of false-positives vs. false-negatives

<br>




### Comparison to Baseline: XGBoost vs Logistic Regression

Compared to a baseline Logistic Regression model, the XGBoost model outperformed the baseline in both the validation and test sets. The higher performance of XGBoost is more notable on the hold-out test set, while on the validation set it can be argued that they are roughly equivalent. The implication is that XGBoost generalizes better on unseen data.



![Precision-Recall](precision_recall.png)

## Workflow
1. Exploratory Data Analysis (EDA)
2. Model Sepcification
3. Hyperparameter Tuning
4. Final Model Training
5. Performance Evaluation on Test Set
6. Deploy model on Vertex AI Endpoint

<br>

___


## EDA and Explanation of Features

There's not a lot to say about this dataset, as most of the columns have been stripped of their original meaning via PCA for anonymization. 

`Class` is the target label, which we will attempt to predict. Definitions for the rest of the features are below:

* `Time`: the time in seconds that the transaction occured, after the first transaction in the dataset
* `Amount`: the transaction value 
* `Class`: binary class label, flagging whether the transaction is suspicious or not
* `V*`: attributes of the transaction, processed with PCA to anonymize the data

In [ ]:
import os
import pandas as pd
import altair as alt
import seaborn as sns

os.listdir('data/')

df = pd.read_csv("data/creditcard.csv")
df.head()

In [ ]:
# Highly imbalanced class labels
df['Class'].value_counts()

In [ ]:
from src.utils import eda_plot1
eda_plot1(df).show()

The charts above shows that fraud activity occurs intermittently (somewhat randomly) over time, but we also see spikes which may suggest events can cluster around certain times. 

## Model Training - XGBoost

Since there are only 30 input features, we remove the `Time` column and skip feature selection, going straight to hyperparameter tuning.

The XGBoost algorithm will automatically split on the most informative features and ignore the rest, so having a larger feature set than necessary doesn't lead to issues like multicollinearity that we would observe in linear regression. However, it does help to narrow down the input features for explainability reasons, when it becomes very large (i.e. more than 100)   



### Hyperparameter Tuning with Optuna

* The cross-validation splits are created using `TimeSeriesSplit` rather than `train_test_split` in order to prevent temporal data leakage
* Using the `optuna` library, we perform a hyperparameter search with the TPE algorithm 
* The important parameters to pay attention to are: 
    * `scale_pos_weight`: the weighting factor on positive labels, to compensate for the imbalanced classes
    * `eval_metric`: we choose the `aucpr` evaluation metric here intentionally, to maximize the area under the precision-recall curve
    * `min_child_weight`: controls the minimum sum of instance weights in a leaf node
    * `max_delta_step`: constrains the magnitude of each tree's weight update, stabilizing training when class distribution is extreme

We initialize our `scale_pos_weight` variable to the ratio of negative classes to positive classes:

$$
\text{scale\_pos\_weight} = \dfrac{\# Negative Labels}{\# Positive Labels}
$$

In [ ]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.pipeline import Pipeline
import shap
from sklearn.metrics import average_precision_score, f1_score
import numpy as np

tscv = TimeSeriesSplit(n_splits=10)
df_ts = df.sort_values(by='Time').reset_index(drop=True)
X_ts, y_ts = df_ts.drop(columns=['Class', 'Time']), df_ts['Class']

print(f"scale_pos_weight starting value: {(y_ts == 0).sum() / (y_ts==1).sum():.4f}")

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 10.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 10.0),
        'min_child_weight': trial.suggest_float('min_child_weight', 0, 10),
        'objective': 'binary:logistic',
        'use_label_encoder': False,
        'verbosity': 0,
        'n_jobs': -1,
        'random_state': 0,
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 500, 2000),
        'eval_metric': 'aucpr',
        'early_stopping_rounds': 30
    }

    scores = []
    for train_idx, valid_idx in list(tscv.split(X_ts))[:-1]:
        X_train, X_valid = X_ts.iloc[train_idx], X_ts.iloc[valid_idx]
        y_train, y_valid = y_ts.iloc[train_idx], y_ts.iloc[valid_idx]

        clf = xgb.XGBClassifier(**params)
        clf.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            #early_stopping_rounds=30,
            verbose=False
        )
        y_pred = clf.predict_proba(X_valid)[:, 1]
        scores.append(average_precision_score(y_valid, y_pred))

    return np.mean(scores)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=30, n_jobs=1)

### Feature Importance

Taking a look at the SHAP summary, we observe that `V14` and `V4` are the two most influential feaures in the XGBoost model. 

* Higher values of `V14` tend to push the model towards predicting a negative label
* Higher values of `V4` tend to push the model towards predicting a positive label

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train)
shap.summary_plot(shap_values, X_train, max_display=20)

### Train Final Model with Best parameters from Hyperparameter Search, on the full Training Set

Using the hyperparameters retrieved from the parameter search, we fit the model onto the full training set:
* The distribution of scores by their actual class label indicates a neat separation. 
* Almost all the legitimate transactions received a score close to 0, while all the transactions with a score greater than 0.2 turned out to be fraudulent.

In [ ]:
print('='*50)
print(f"Best PRC Score: {study.best_value:.4f}")
print("Best Parameters: ")
for k, v in study.best_params.items():
    print(f"\t {k}: {v}")
print('='*50)

import pickle

if 'xgb_creditcard.pkl' in os.listdir('models/'):
    with open('models/xgb_creditcard.pkl', 'rb') as file:
        model = pickle.load(file)
else:
    train_idx, valid_idx = list(tscv.split(X_ts))[-2]
    _, test_idx = list(tscv.split(X_ts))[-1]

    X_train, y_train = X_ts.iloc[train_idx], y_ts.iloc[train_idx] 
    X_valid, y_valid = X_ts.iloc[valid_idx], y_ts.iloc[valid_idx]
    X_test, y_test = X_ts.iloc[test_idx], y_ts.iloc[test_idx]

    model = xgb.XGBClassifier(**study.best_params)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=100)

    import pickle
    with open('models/xgb_creditcard.pkl', 'wb') as file:
        # Save model
        pickle.dump(model, file)

y_pred = model.predict_proba(X_valid)[:,1]
y_pred

In [ ]:
from src.utils import score_distribution_plot
score_distribution_plot(y_pred, y_valid)
#import seaborn as sns
#plot_df = pd.DataFrame({
#    'score': y_pred,
#    'class': y_valid
#})
#plot_df['label'] = np.where(plot_df['class']==1, 'Fraud', 'Legitimate')
#fig, ax = plt.subplots(figsize=(10, 6))
#sns.histplot(data=plot_df, x='score', hue='label', stat='probability', common_norm=False, bins=50, alpha=0.5, ax=ax)
#ax.set_xlabel("Predicted Score")
#ax.set_title("Score Distribution by Class (Validation Set)")


### Precision-Recall Curve

The precision-recall curve shows that we can achieve a precision above 90% and a recall above 80%, on the validation set. 

The shape of the cuvre indicates a steap drop-off in precision at around the recall=0.82 mark, which should be kept in mind.

In [ ]:
from src.utils import precision_recall_chart1
y_pred = model.predict_proba(X_valid)[:,1]
print('='*50 + '\n')
print(f"\tAUCPRC: {average_precision_score(y_valid, y_pred):.4f}\n")
print('='*50 + '\n')
precision_recall_chart1(model, X_valid, y_valid)

## Cost-Minimization Approach to Selecting the Classifier Threshold Value

Since fraud detection is inherently a problem with an asymmetric cost structure, we select the classifier threshold such that the total cost to the business is minimized. 
* In general, the cost of a false negative (i.e. failing to detect fraud) is usually much higher than the cost of a false positive (i.e. flagging a legitimate transaction as fraud). 
* In the former case, a fraudulent charge can result in chargebacks, lost merchandise, and sometimes investigative fees. 
* In the latter case, an inadvertently denied transaction that turns out to be legitimate results in a missed sales opportunity and possibly churn.  



### Methodology

* Let $C_{FN}$ and $C_{FP}$ be the cost for a false-negative and false-positive respectively
* Let $FN(t)$ and $FP(t)$ be the number of occurrences of a false-negative and false-positive at time $(t)$ respectively
* The cost to the business at time $(t)$ can be represented as below

$$
Cost(t)  = FN(t) \cdot C_{FN} + FP(t) \cdot C_{FP}
$$


* After discussing with our business partners, we agreed on the values below to set for $C_{FN}$ and $C_{FP}$.

$$
C_{FN} = \$150 \\

C_{FP} = \$12
$$

* For each threshold value, we calculate the total cost using the precision and recall arrays and store in a list
* Find the threshold value which corresponds to the lowest cost

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

# Define costs — get these from the business
COST_FALSE_NEGATIVE = 150 # avg fraud loss + chargeback fee per missed fraud
COST_FALSE_POSITIVE = 12 # estimated friction cost per false decline

y_pred = model.predict_proba(X_valid)[:,1]
precisions, recalls, thresholds = precision_recall_curve(y_valid, y_pred)
p, r = precisions[:-1], recalls[:-1]

# Total positives and negatives in val set
n_pos = y_valid.sum()
n_neg = (y_valid == 0).sum()

costs = []
for i, t in enumerate(thresholds):
    if p[i] == 0:
        costs.append(float('inf'))
        continue
    # At threshold t:
    fn = n_pos * (1 - r[i])          # missed fraud cases
    tp = r[i] * n_pos
    fp = tp * (1 - p[i]) / p[i]        # false positives implied by precision
    cost = COST_FALSE_NEGATIVE * fn + COST_FALSE_POSITIVE * fp
    costs.append(cost)

best_idx      = np.argmin(costs)
best_threshold = thresholds[best_idx]
best_cost      = costs[best_idx]

print(f"Optimal threshold: {best_threshold:.3f}")
print(f"Precision at threshold: {precisions[best_idx]:.3f}")
print(f"Recall at threshold:    {recalls[best_idx]:.3f}")
print(f"Expected cost:          ${best_cost:,.0f}")

### Confusion Matrix - Validation Set

On the validation set, using the classifier $(score >= 0.236)$ yields the confusion matrix below. 

> Notably, the model catches 40 (out of 49) actual fraud transactions, while only generating 1 false-negative.

In [ ]:
from src.utils import confusion_matrix_chart1
plot_df = pd.DataFrame({
    'score': y_pred,
    'actual': y_valid,
})
plot_df['predicted_class'] = np.where(plot_df['score']>=best_threshold, 1, 0)
confusion_matrix_chart1(plot_df, best_threshold)

## Evaluate the final model on the holdout test set

Evaluation of the model on the holdout test set provides a perspective on how the model will perform on unseen data.

The confusion matrix shows that the model had 15 true-positives, 6 false-negatives, and 13 false-positives. 

The lower count of false-negatives as opposed to false-positives indicates successful parameterization, reflecting the preference of minimizing uncaught fraud over declined legitimate transactions. 

### Comparison to baseline Logistic Regression model

* AUCPR
* AUCROC


In [ ]:
y_pred_test = model.predict_proba(X_test)[:, 1]
plot_df = pd.DataFrame({
    'score': y_pred_test,
    'actual': y_test,
})
plot_df['predicted_class'] = np.where(plot_df['score']>=best_threshold, 1, 0)
confusion_matrix_chart1(plot_df, best_threshold)

* Precision-Recall Curve (Validation vs Test)

## Deploy Model on Vertex AI 

### Extra

In [ ]:
from sklearn.metrics import precision_recall_curve, PrecisionRecallDisplay, ConfusionMatrixDisplay, confusion_matrix

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
plot_df = pd.DataFrame({
    'score': y_pred,
    'actual': y_valid,
})
plot_df['predicted_class'] = np.where(plot_df['score']>=best_threshold, 1, 0)
cm = confusion_matrix(plot_df['actual'], plot_df['predicted_class'])
disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Legitimate', 'Fraud']
    )
disp.plot(cmap='Blues', colorbar=False, ax=ax[0])
ax[0].set_title(f"Validation Set (Threshold={best_threshold:.2f})", size=12)

y_pred_test = model.predict_proba(X_test)[:, 1]
plot_df = pd.DataFrame({
    'score': y_pred_test,
    'actual': y_test,
})
plot_df['predicted_class'] = np.where(plot_df['score']>=best_threshold, 1, 0)
cm = confusion_matrix(plot_df['actual'], plot_df['predicted_class'])
disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Legitimate', 'Fraud']
    )
disp.plot(cmap='Blues', colorbar=False, ax=ax[1])
ax[1].set_title(f"Test Set (Threshold={best_threshold:.2f})", size=12)
ax[1].set_ylabel('')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100)

## Logistic Regression for Baseline Modeling

In [ ]:
from sklearn.linear_model import LogisticRegression
import optuna

def objective(trial):
    
    penalty = trial.suggest_categorical("penalty", ["l1", "l2", "elasticnet"])

    if penalty == 'elasticnet':
        l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)
    else:
        l1_ratio = None

    if penalty == 'elasticnet':
        solver = 'saga'
    elif penalty == 'l1':
        solver = 'saga'
    else:
        solver = 'lbfgs'

    C = trial.suggest_float("C", 1e-3, 1e2, log=True)

    scores = []
    for train_idx, valid_idx in list(tscv.split(X_ts))[:-1]:
        X_train, X_valid = X_ts.iloc[train_idx], X_ts.iloc[valid_idx]
        y_train, y_valid = y_ts.iloc[train_idx], y_ts.iloc[valid_idx]

        model = LogisticRegression(
            penalty = penalty,
            C=C,
            l1_ratio=l1_ratio,
            solver=solver,
            class_weight='balanced',
            max_iter=1000,
            random_state=0,
            #scoring='average_precision'
        )
        model.fit(X_train, y_train)
        y_hat = model.predict_proba(X_valid)[:,1]
        scores.append(average_precision_score(y_valid, y_hat))
    
    return np.mean(scores)


study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=30, show_progress_bar=True)

best = study.best_params
logistic_model = LogisticRegression(
    penalty = best['penalty'],
    C=best['C'],
    l1_ratio = best.get('l1_ratio'),
    solver='saga' if best['penalty'] in ('l1', 'elasticnet') else 'lbfgs',
    class_weight = 'balanced',
    max_iter=1000,
    random_state=0
)
logistic_model.fit(X_train, y_train)

with open('models/logistic.pkl', 'wb') as file:
    pickle.dump(model, file)


In [ ]:
print(f"\nBest AUC-PR: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(10, 6))
display = PrecisionRecallDisplay.from_estimator(model, X_valid, y_valid, ax=ax[0])
display.ax_.set_title("Precision-Recall Curve (Validation Set)")
display = PrecisionRecallDisplay.from_estimator(logistic_model, X_valid, y_valid, ax=ax[0])

display = PrecisionRecallDisplay.from_estimator(model, X_test, y_test, ax=ax[1])
display.ax_.set_title("Precision-Recall Curve (Test Set)")
display = PrecisionRecallDisplay.from_estimator(logistic_model, X_test, y_test, ax=ax[1])

plt.tight_layout()
plt.savefig("precision_recall.png")